In [1]:
import pandas as pd

# Đọc dữ liệu gốc từ thư mục raw
df = pd.read_csv('../data/raw/depression_severity_dataset.csv')

print("Số dòng:", len(df))
print("Số cột:", len(df.columns))
print("Tên cột:", list(df.columns))

Số dòng: 3553
Số cột: 2
Tên cột: ['text', 'label']


In [2]:
print("Phân bố nhãn:")
print(df['label'].value_counts())

print("\nTỉ lệ phần trăm:")
print((df['label'].value_counts(normalize=True) * 100).round(2))

Phân bố nhãn:
label
minimum     2587
moderate     394
mild         290
severe       282
Name: count, dtype: int64

Tỉ lệ phần trăm:
label
minimum     72.81
moderate    11.09
mild         8.16
severe       7.94
Name: proportion, dtype: float64


In [3]:
import re

# Tìm các dòng bị lỗi dạng #NAME?, #REF!, #VALUE!...
excel_error_pattern = r'^#(NAME|REF|VALUE|DIV|N/A|NULL)\??!?$'
mask_excel_error = df['text'].astype(str).str.match(excel_error_pattern, na=False)

print("Số dòng bị lỗi Excel:", mask_excel_error.sum())
print("\nCác dòng bị lỗi:")
print(df[mask_excel_error])

Số dòng bị lỗi Excel: 4

Các dòng bị lỗi:
        text    label
295   #NAME?  minimum
1592  #NAME?  minimum
2190  #NAME?  minimum
2563  #NAME?  minimum


In [4]:
n_before = len(df)
df = df[~mask_excel_error].copy()
n_after = len(df)

print(f"Trước khi loại: {n_before} dòng")
print(f"Sau khi loại:   {n_after} dòng")
print(f"Đã loại:        {n_before - n_after} dòng")

Trước khi loại: 3553 dòng
Sau khi loại:   3549 dòng
Đã loại:        4 dòng


In [5]:
print("Số giá trị null trong từng cột:")
print(df.isnull().sum())

print("\nSố dòng có text chỉ chứa khoảng trắng:")
print((df['text'].astype(str).str.strip() == '').sum())

Số giá trị null trong từng cột:
text     0
label    0
dtype: int64

Số dòng có text chỉ chứa khoảng trắng:
0


In [6]:
def normalize_whitespace(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)          # nhiều khoảng trắng liên tiếp -> 1 khoảng trắng
    text = re.sub(r'[\r\n\t]+', ' ', text)    # xóa ký tự xuống dòng / tab
    return text.strip()

df['text'] = df['text'].apply(normalize_whitespace)

print("Đã chuẩn hóa xong.")
print("Ví dụ 3 dòng đầu sau khi chuẩn hóa:")
print(df['text'].head(3))

Đã chuẩn hóa xong.
Ví dụ 3 dòng đầu sau khi chuẩn hóa:
0    He said he had not felt that way before, sugge...
1    Hey there r/assistance, Not sure if this is th...
2    My mom then hit me with the newspaper and it s...
Name: text, dtype: str


In [7]:
n_before = len(df)
n_dup_exact = df.duplicated(subset=['text']).sum()

print(f"Số dòng trùng lặp (nội dung text giống hệt nhau): {n_dup_exact}")

df = df.drop_duplicates(subset=['text'], keep='first').copy()
n_after = len(df)

print(f"\nTrước khi loại: {n_before} dòng")
print(f"Sau khi loại:   {n_after} dòng")
print(f"Đã loại:        {n_before - n_after} dòng")

Số dòng trùng lặp (nội dung text giống hệt nhau): 18

Trước khi loại: 3549 dòng
Sau khi loại:   3531 dòng
Đã loại:        18 dòng


In [8]:
df['word_count'] = df['text'].str.split().str.len()

print("Thống kê độ dài (số từ) của bài đăng:")
print(df['word_count'].describe())

mask_too_short = df['word_count'] < 3
print(f"\nSố bài đăng dưới 3 từ: {mask_too_short.sum()}")
print(df[mask_too_short])

Thống kê độ dài (số từ) của bài đăng:
count    3531.000000
mean       85.672897
std        31.981019
min         6.000000
25%        65.000000
50%        80.000000
75%       101.000000
max       310.000000
Name: word_count, dtype: float64

Số bài đăng dưới 3 từ: 0
Empty DataFrame
Columns: [text, label, word_count]
Index: []


In [9]:
print("=" * 50)
print("TỔNG KẾT DATA CLEANING")
print("=" * 50)
print("Số dòng ban đầu:  3553")
print(f"Số dòng còn lại:  {len(df)}")
print(f"Đã loại tổng cộng: {3553 - len(df)} dòng ({(3553-len(df))/3553*100:.2f}%)")

print("\nPhân bố nhãn sau khi làm sạch:")
print(df['label'].value_counts())
print("\nTỉ lệ phần trăm:")
print((df['label'].value_counts(normalize=True) * 100).round(2))

# Lưu file đã làm sạch vào data/processed/
df_out = df[['text', 'label']].reset_index(drop=True)
df_out.to_csv('../data/processed/depression_severity_cleaned.csv', index=False)

print(f"\nĐã lưu file sạch: data/processed/depression_severity_cleaned.csv ({len(df_out)} dòng)")

TỔNG KẾT DATA CLEANING
Số dòng ban đầu:  3553
Số dòng còn lại:  3531
Đã loại tổng cộng: 22 dòng (0.62%)

Phân bố nhãn sau khi làm sạch:
label
minimum     2566
moderate     394
mild         290
severe       281
Name: count, dtype: int64

Tỉ lệ phần trăm:
label
minimum     72.67
moderate    11.16
mild         8.21
severe       7.96
Name: proportion, dtype: float64

Đã lưu file sạch: data/processed/depression_severity_cleaned.csv (3531 dòng)
